# Huberman Lab Youtube Extraction

### Setup

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import os

In [6]:
# Get playlist page using Selenium since the content is dynamically loaded
driver = webdriver.Chrome()  # Make sure you have ChromeDriver installed
driver.get("https://www.youtube.com/playlist?list=PLPNW_gerXa4Pc8S2qoUQc5e8Ir97RLuVW")

# Function to scroll to bottom of page
def scroll_to_bottom():
    last_height = driver.execute_script("return document.documentElement.scrollHeight")
    while True:
        # Scroll down
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        # Wait for new videos to load
        time.sleep(2)
        # Calculate new scroll height
        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

# Scroll to load all videos
scroll_to_bottom()

# Wait for the video elements to load
wait = WebDriverWait(driver, 30)
video_elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ytd-playlist-video-renderer")))

# Extract video information
video_data = []
for element in video_elements:
    # Get URL and title
    video_link = element.find_element(By.CSS_SELECTOR, "a#video-title").get_attribute('href')
    title = element.find_element(By.CSS_SELECTOR, "a#video-title").text
    
    if video_link and '/watch?v=' in video_link:
        video_data.append({
            'url': video_link,
            'title': title
        })

driver.quit()

# Create DataFrame
df_videos = pd.DataFrame(video_data)
df_videos.head()


,url,title
0,https://www.youtube.com/watch?v=_eDg9yOvvrQ&li...,"How Smell, Taste & Pheromones Shape Behavior |..."
1,https://www.youtube.com/watch?v=6ck9fa6_C8c&li...,What Pets Actually Want & Need | Dr. Karolina ...
2,https://www.youtube.com/watch?v=q-wRvsiGYIs&li...,"AMA #19: Collagen vs. Whey Protein, Creatine, ..."
3,https://www.youtube.com/watch?v=ssmwxKPFMFU&li...,Protocols to Improve Vision & Eyesight | Huber...
4,https://www.youtube.com/watch?v=J7yn4tJEmJU&li...,Tools for Overcoming Substance & Behavioral Ad...


In [7]:
# Extract video key from URL by splitting on '=' and taking everything after it
df_videos['video_key'] = df_videos['url'].str.split('=').str[1].str.split('&').str[0]
df_videos


,url,title,video_id
0,https://www.youtube.com/watch?v=_eDg9yOvvrQ&li...,"How Smell, Taste & Pheromones Shape Behavior |...",_eDg9yOvvrQ
1,https://www.youtube.com/watch?v=6ck9fa6_C8c&li...,What Pets Actually Want & Need | Dr. Karolina ...,6ck9fa6_C8c
2,https://www.youtube.com/watch?v=q-wRvsiGYIs&li...,"AMA #19: Collagen vs. Whey Protein, Creatine, ...",q-wRvsiGYIs
3,https://www.youtube.com/watch?v=ssmwxKPFMFU&li...,Protocols to Improve Vision & Eyesight | Huber...,ssmwxKPFMFU
4,https://www.youtube.com/watch?v=J7yn4tJEmJU&li...,Tools for Overcoming Substance & Behavioral Ad...,J7yn4tJEmJU
...,...,...,...
280,https://www.youtube.com/watch?v=NAATB55oxeQ&li...,"How to Defeat Jet Lag, Shift Work & Sleeplessness",NAATB55oxeQ
281,https://www.youtube.com/watch?v=nwSkFq4tyC0&li...,"Using Science to Optimize Sleep, Learning & Me...",nwSkFq4tyC0
282,https://www.youtube.com/watch?v=nm1TxQj9IsQ&li...,Master Your Sleep & Be More Alert When Awake,nm1TxQj9IsQ
283,https://www.youtube.com/watch?v=H-XfCl-HpRM&li...,How Your Brain Works & Changes,H-XfCl-HpRM


In [9]:
os.makedirs('data', exist_ok=True)
df_videos.to_csv('data/huberman_videos.csv', index=False)